# Hi-RAG reproduction: Phase 2, baselines with an LLM picking the tool

Methods: **Flat-RAG**, **ColBERT-RAG**, **Stage 1 only**, **Full context**. LLM via **Groq** (free tier).

Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi -L

## 1. Code + install
Upload the new `hirag-repro.zip`.

In [ ]:
%cd /content
from google.colab import files
up = files.upload()                      # choose hirag-repro.zip
name = list(up.keys())[0]                # works even if Colab renames it 'hirag-repro (1).zip'
!rm -rf /content/hirag-repro
!unzip -q -o "/content/{name}" -d /content/
%cd /content/hirag-repro
!pip install -q -r requirements.txt

## 2. Keep the LLM cache on Google Drive
If Colab disconnects or you hit the daily limit, re-running continues where it stopped (answers already received are not requested again).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CACHE = '/content/drive/MyDrive/hirag_llm_cache.jsonl'

## 3. API key (typed in, never saved in the notebook)

In [ ]:
import os, getpass
os.environ['LLM_API_KEY'] = getpass.getpass('Groq API key (gsk_...): ')

## 4. Which models does your key have?
The paper used Qwen3-32B, which Groq retired on 2026-07-17. Default here is `qwen/qwen3.8-27b`; if it is not in the list, pick another (e.g. `openai/gpt-oss-20b`) and change `MODEL` below.

In [ ]:
from openai import OpenAI
c = OpenAI(api_key=os.environ['LLM_API_KEY'], base_url='https://api.groq.com/openai/v1')
print(sorted(m.id for m in c.models.list().data))

In [ ]:
MODEL = 'qwen/qwen3.8-27b'
EXTRA = '{"reasoning_effort": "none", "reasoning_format": "hidden"}'   # for Qwen; use '{}' for other models
COMMON = (f'--model {MODEL} --cache {CACHE} --embed-model BAAI/bge-large-en --max-tokens 128 '
          f'--out-dir /content/drive/MyDrive/hirag_results')
print(COMMON)

### Limit check (optional)

In [ ]:
# How much of today's Groq limit is left? (run this if a run seems stuck)
c = OpenAI(api_key=os.environ['LLM_API_KEY'], base_url='https://api.groq.com/openai/v1')
try:
    r = c.chat.completions.with_raw_response.create(model=MODEL, messages=[{'role': 'user', 'content': 'hi'}],
                                                   max_tokens=5, extra_body={'reasoning_effort': 'none'})
    for h, v in r.headers.items():
        if 'ratelimit' in h or 'retry' in h:
            print(h, '=', v)
    print('OK: the key works right now')
except Exception as e:
    print('ERROR:', e)

## 5. Quick test on 5 queries

In [ ]:
!python scripts/run_e2e.py --method flat --split single --limit 5 {COMMON} --extra-body '{EXTRA}'

## 6. Full runs
Groq's free plan allows only **1,000 output tokens per minute**, so the script waits about 15 s between calls (`--max-tokens 256`). A single-service run (201 calls) takes about **50 minutes**; a multi-service run about **30–40 minutes**. Keep the tab open. If you hit the **daily** limit, continue tomorrow (the cache keeps finished work).

In [ ]:
!python scripts/run_e2e.py --method flat   --split single --k 1 {COMMON} --extra-body '{EXTRA}'
!python scripts/run_e2e.py --method flat   --split multi  --k 3 {COMMON} --extra-body '{EXTRA}'

In [ ]:
!python scripts/run_e2e.py --method stage1 --split single --k 1 {COMMON} --extra-body '{EXTRA}'
!python scripts/run_e2e.py --method stage1 --split multi  --k 3 {COMMON} --extra-body '{EXTRA}'

In [ ]:
!python scripts/run_e2e.py --method colbert --split single --k 1 {COMMON} --extra-body '{EXTRA}'
!python scripts/run_e2e.py --method colbert --split multi  --k 3 {COMMON} --extra-body '{EXTRA}'

### Full context (all 201 tools, about 15,000 prompt tokens per call)
This may exceed the free tier's tokens-per-minute limit. Try 5 queries first; if you get 'Request too large' or constant rate-limit waits, skip it and tell me.

In [ ]:
!python scripts/run_e2e.py --method full --split single --limit 5 --min-interval 20 {COMMON} --extra-body '{EXTRA}'

## 7. Download results

In [ ]:
!cd /content/drive/MyDrive && zip -q -r /content/results.zip hirag_results
files.download('/content/results.zip')

## 8. Option: local model with Ollama (free, no limits)
Runs Qwen on the Colab T4 GPU. Smaller model than Groq's, so numbers differ; report it as a separate row.
Run **section 1 (upload) and 2 (Drive)** first; the API-key cell is not needed.

In [ ]:
!sudo apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time
subprocess.Popen('nohup ollama serve > /content/ollama.log 2>&1', shell=True)
time.sleep(5)
!ollama --version

In [ ]:
LOCAL_MODEL = 'qwen3:14b'     # ~9 GB, fits the T4; if too slow use 'qwen3:8b'
!ollama pull {LOCAL_MODEL}

In [ ]:
LOCAL = (f'--provider ollama --model {LOCAL_MODEL} --min-interval 0 --max-tokens 256 '
         f'--cache /content/drive/MyDrive/hirag_llm_cache_local.jsonl --embed-model BAAI/bge-large-en '
         f'--out-dir /content/drive/MyDrive/hirag_results_local')
LOCAL_EXTRA = '{"reasoning_effort": "none"}'   # turn Qwen3 thinking off
print(LOCAL)

In [ ]:
# quick test first: 5 queries
!python scripts/run_e2e.py --method flat --split single --limit 5 {LOCAL} --extra-body '{LOCAL_EXTRA}'

In [ ]:
!python scripts/run_e2e.py --method flat    --split single --k 1 {LOCAL} --extra-body '{LOCAL_EXTRA}'
!python scripts/run_e2e.py --method stage1  --split single --k 1 {LOCAL} --extra-body '{LOCAL_EXTRA}'
!python scripts/run_e2e.py --method colbert --split single --k 1 {LOCAL} --extra-body '{LOCAL_EXTRA}'

In [ ]:
!python scripts/run_e2e.py --method flat    --split multi --k 3 {LOCAL} --extra-body '{LOCAL_EXTRA}'
!python scripts/run_e2e.py --method stage1  --split multi --k 3 {LOCAL} --extra-body '{LOCAL_EXTRA}'
!python scripts/run_e2e.py --method colbert --split multi --k 3 {LOCAL} --extra-body '{LOCAL_EXTRA}'